### <span style="color:lightgreen"> Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)   </span>

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from scikeras.wrappers import KerasClassifier


# Load and preprocess data
try:
    data = pd.read_csv('Churn_Modelling.csv')
    data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
except FileNotFoundError as e:
    print(f"Error: {e}")
    exit()

# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

# Split features and target
x = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save preprocessing objects
try:
    with open('gender_encoder.pkl', 'wb') as file:
        pickle.dump(label_encoder_gender, file)
    with open('geography_encoder.pkl', 'wb') as file:
        pickle.dump(onehot_encoder_geo, file)
    with open('dataScaler.pkl', 'wb') as file:
        pickle.dump(scaler, file)
except Exception as e:
    print(f"Error during serialization: {e}")


# Define model
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', kernel_initializer='he_uniform', input_shape=(X_train.shape[1],)))

    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu', kernel_initializer='he_uniform'))
        model.add(Dropout(0.3))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


# Perform hyperparameter tuning on a neural network model
model = KerasClassifier(build_fn=create_model, verbose=1)

# Grid Search for Hyperparameter Tuning
param_grid = {
    'model__neurons': [16, 32, 64],  # Pass neurons to build_fn
    'model__layers': [1, 2],             # Pass layers to build_fn
    'epochs': [50, 100],                    # Passed directly to KerasClassifier
}

# GridSearchCV Initialization
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")


Fitting 3 folds for each of 18 candidates, totalling 54 fits
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argu

167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6808 - loss: 0.6884
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7358 - loss: 0.6142
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6836 - loss: 0.7214
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7188 - loss: 0.6937
104/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7523 - loss: 0.6318Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6997 - loss: 0.7338
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7424 - loss: 0.6700 
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7266 - loss: 0.6414 
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7608 - loss: 0.5957 
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7705 - loss: 0.5448
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7882 - loss: 0.5164
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7862 - l

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8937 - loss: 0.2554 
Epoch 48/50
 1/84 ━━━━━━━━━━━━━━━━━━━━ 7s 90ms/stepEpoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8877 - loss: 0.2507
Epoch 39/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/stepep - accuracy: 0.8605 - loss: 0.2821
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8717 - loss: 0.3098
Epoch 48/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8889 - loss: 0.2441
Epoch 41/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8671 - loss: 0.2799
Epoch 47/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8987 - loss: 0.2409 
Epoch 47/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8883 - loss: 0.2627
 75/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8695 - loss: 0.3045Epoch 49/50
156/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8980 - loss: 0.2372Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8979 - loss: 0.2372
Epoch 40/50
 35/167 ━━━━━━━━━━

...

 92/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8609 - loss: 0.3357Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8496 - loss: 0.3699
Epoch 16/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/stepep - accuracy: 0.8916 - loss: 0.2644
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8640 - loss: 0.3289
Epoch 91/100
  1/167 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8750 - loss: 0.3225Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9415 - loss: 0.1334
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8880 - loss: 0.2698
Epoch 58/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9349 - loss: 0.1449
Epoch 99/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8623 - loss: 0.3413
Epoch 17/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/stepep - accuracy: 0.8939 - loss: 0.2557
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8691 - loss: 0.3153
Epoch 92/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7251 - loss: 0.6838

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8491 - loss: 0.3605
Epoch 18/100
115/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9310 - loss: 0.1573Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7864 - loss: 0.5218
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8930 - loss: 0.2489
Epoch 60/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9329 - loss: 0.1545
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8752 - loss: 0.3056
Epoch 93/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8644 - loss: 0.3434
Epoch 19/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/stepccuracy: 0.7812 - loss: 0.38: 0.47
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7917 - loss: 0.4743
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8696 - loss: 0.3105
Epoch 94/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8561 - loss: 0.3520
Epoch 20/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8972 - lo

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
1 fits failed out of a total of 54.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py", line 1501, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/scikeras/wrappers.py", line 770, in fit
    self._fit(
 

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 623us/step - accuracy: 0.7680 - loss: 0.6368
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 517us/step - accuracy: 0.7874 - loss: 0.5290
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 522us/step - accuracy: 0.8037 - loss: 0.4809
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 503us/step - accuracy: 0.7955 - loss: 0.4666
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - accuracy: 0.7971 - loss: 0.4614
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 542us/step - accuracy: 0.8166 - loss: 0.4361
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - accuracy: 0.8191 - loss: 0.4255
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step - accuracy: 0.8236 - loss: 0.4129
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 511us/step - accuracy: 0.8151 - loss: 0.4203
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 499us/step - accuracy: 0.8211 - loss: 0.4162
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 645us/step - accuracy: 0.8372 - loss: 0.3883
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 